In [1]:
!pip install spacy transformers torch pandas seqeval
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=5aa06af4a17ad14aece989811d7870e7dbca87618b5e88f7ea10984f43e912fc
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 76.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# spaCy library for NLP processing and NER
import spacy

# displacy for visualization of entities
from spacy import displacy

# Transformers for pretrained NER models
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# torch for deep learning backend used by transformers
import torch

# pandas for tabular data storage
import pandas as pd

In [3]:
# Load English pretrained model
nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully")

spaCy model loaded successfully


In [4]:
sentences = [
    "India won the cricket match against Australia in Mumbai.",
    "Narendra Modi met Joe Biden in Washington.",
    "Microsoft announced a new AI model in 2025.",
    "Elon Musk is the CEO of Tesla and SpaceX.",
    "The Olympics will be held in Paris."
]

In [5]:
spacy_results = []

for sent in sentences:
    doc = nlp(sent)

    print("\nSentence:", sent)
    print("Entities:")

    for ent in doc.ents:
        print(ent.text, "->", ent.label_)

        spacy_results.append([sent, ent.text, ent.label_])


Sentence: India won the cricket match against Australia in Mumbai.
Entities:
India -> GPE
Australia -> GPE
Mumbai -> GPE

Sentence: Narendra Modi met Joe Biden in Washington.
Entities:
Narendra Modi -> PERSON
Joe Biden -> PERSON
Washington -> GPE

Sentence: Microsoft announced a new AI model in 2025.
Entities:
Microsoft -> ORG
AI -> GPE
2025 -> DATE

Sentence: Elon Musk is the CEO of Tesla and SpaceX.
Entities:
Elon Musk -> PERSON
Tesla -> ORG

Sentence: The Olympics will be held in Paris.
Entities:
Olympics -> EVENT
Paris -> GPE


In [6]:
df_spacy = pd.DataFrame(spacy_results, columns=["Sentence", "Entity", "Label"])

df_spacy

,Sentence,Entity,Label
0,India won the cricket match against Australia ...,India,GPE
1,India won the cricket match against Australia ...,Australia,GPE
2,India won the cricket match against Australia ...,Mumbai,GPE
3,Narendra Modi met Joe Biden in Washington.,Narendra Modi,PERSON
4,Narendra Modi met Joe Biden in Washington.,Joe Biden,PERSON
5,Narendra Modi met Joe Biden in Washington.,Washington,GPE
6,Microsoft announced a new AI model in 2025.,Microsoft,ORG
7,Microsoft announced a new AI model in 2025.,AI,GPE
8,Microsoft announced a new AI model in 2025.,2025,DATE
9,Elon Musk is the CEO of Tesla and SpaceX.,Elon Musk,PERSON


In [7]:
doc = nlp(sentences[1])
displacy.render(doc, style="ent", jupyter=True)

In [8]:
model_name = "dbmdz/bert-large-cased-finetuned-conll03-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

print("Transformer model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transformer model loaded


In [9]:
transformer_results = []

for sent in sentences:
    print("\nSentence:", sent)

    output = ner_pipeline(sent)

    for entity in output:
        word = entity['word']
        label = entity['entity_group']
        score = entity['score']

        print(word, "->", label, "| Confidence:", round(score,3))

        transformer_results.append([sent, word, label, score])


Sentence: India won the cricket match against Australia in Mumbai.
India -> LOC | Confidence: 1.0
Australia -> LOC | Confidence: 1.0
Mumbai -> LOC | Confidence: 0.999

Sentence: Narendra Modi met Joe Biden in Washington.
Narendra Modi -> PER | Confidence: 0.994
Joe Biden -> PER | Confidence: 0.997
Washington -> LOC | Confidence: 0.999

Sentence: Microsoft announced a new AI model in 2025.
Microsoft -> ORG | Confidence: 0.999

Sentence: Elon Musk is the CEO of Tesla and SpaceX.
Elon Musk -> PER | Confidence: 0.999
Tesla -> ORG | Confidence: 0.997
SpaceX -> ORG | Confidence: 0.999

Sentence: The Olympics will be held in Paris.
Olympics -> MISC | Confidence: 0.992
Paris -> LOC | Confidence: 0.999


In [10]:
clean_results = []

for sent, word, label, score in transformer_results:

    # remove subword symbols
    word = word.replace("##", "")

    clean_results.append([sent, word, label, score])

df_transformer = pd.DataFrame(
    clean_results,
    columns=["Sentence", "Entity", "Label", "Confidence"]
)

df_transformer

,Sentence,Entity,Label,Confidence
0,India won the cricket match against Australia ...,India,LOC,0.999752
1,India won the cricket match against Australia ...,Australia,LOC,0.999790
2,India won the cricket match against Australia ...,Mumbai,LOC,0.999375
3,Narendra Modi met Joe Biden in Washington.,Narendra Modi,PER,0.993933
4,Narendra Modi met Joe Biden in Washington.,Joe Biden,PER,0.996672
5,Narendra Modi met Joe Biden in Washington.,Washington,LOC,0.999126
6,Microsoft announced a new AI model in 2025.,Microsoft,ORG,0.999218
7,Elon Musk is the CEO of Tesla and SpaceX.,Elon Musk,PER,0.999065
8,Elon Musk is the CEO of Tesla and SpaceX.,Tesla,ORG,0.996662
9,Elon Musk is the CEO of Tesla and SpaceX.,SpaceX,ORG,0.998991


In [11]:
comparison = pd.DataFrame({
    "Feature": [
        "Model Type",
        "Speed",
        "Accuracy",
        "Context Handling",
        "Confidence Score"
    ],
    "spaCy": [
        "Statistical",
        "Fast",
        "Medium",
        "Moderate",
        "No"
    ],
    "HuggingFace": [
        "Transformer",
        "Slower",
        "High",
        "Excellent",
        "Yes"
    ]
})

comparison

,Feature,spaCy,HuggingFace
0,Model Type,Statistical,Transformer
1,Speed,Fast,Slower
2,Accuracy,Medium,High
3,Context Handling,Moderate,Excellent
4,Confidence Score,No,Yes
